In [1]:
!python --version

Python 3.11.14


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [3]:
root0 = '../data'
root0 = Path(root0)

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


['cancer', 'reactome', 'vector_store', 'TCGA', 'gdc_programs.txt']

In [4]:
os.listdir(gdc.root_data)[:10]

['cancer', 'reactome', 'vector_store', 'TCGA', 'gdc_programs.txt']

### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

np.array(prog_list)


File read at './../data/gdc_programs.txt'


array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
prog_id = 'TCGA'

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.head(3)

Table opened ((33, 5)) at '../data/TCGA/primary_site_program_TCGA.tsv'
33


,psi_id,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma


In [8]:
term = 'TCGA_UCEC'

gdc.df_psi[gdc.df_psi.psi_id == 'TCGA-GBM']

,psi_id,primary_site,project_id,disease_type,name
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme


In [9]:
i = 3

psi_id = df_psi.iloc[i].psi_id
gdc.set_primary_site(psi_id=psi_id)

gdc.psi_id, gdc.primary_site, gdc.disease_type, gdc.disease_name, gdc.root_psi

('TCGA-LGG', 'Brain', 'Gliomas', 3, PosixPath('../data/TCGA/TCGA-LGG'))

In [10]:
primary_site = df_psi.iloc[i].primary_site
gdc.set_primary_site(primary_site=primary_site)

gdc.psi_id, gdc.primary_site, gdc.disease_type, gdc.disease_name, gdc.root_psi

('TCGA-LGG', 'Brain', 'Gliomas', 3, PosixPath('../data/TCGA/TCGA-LGG'))

In [ ]:
for i, row in df_psi.iterrows():
    primary_site = row.primary_site
    print(">>>", primary_site)

    df_cases, df_samples, df_mut, barcode_list = gdc.get_filtered_tables(primary_site=primary_site)

    print('-->', len(df_cases), len(df_samples), len(df_mut), len(barcode_list), '\n')

>>> Adrenal gland
No mutation analysis file found for: TCGA-ACC_Adrenal_gland_subtype_sarcoma_tumor_sarcoma_subtype_tissue_sarcoma
Adrenal gland 90 5453 10778 51
>>> Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura
Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura 3 151 26 3
>>> Bladder
Bladder 34 7988 41267 34
>>> Brain
No cases found for primary site: Brain
No samples found for primary site: Brain
No mutations found for primary site: Brain
No barcodes found for primary site: Brain
Brain 0 0 0 0
>>> Brain
No cases found for primary site: Brai

In [ ]:
gdc.s_case

In [ ]:
gdc.df_cases

In [ ]:

def move_tables(primary_site: str,  verbose: bool=False):

    dfa = gdc.df_psi[gdc.df_psi.primary_site == primary_site]

    if dfa.empty:
        gdc.psi_id = ''
        print("No primary site information found for:", primary_site)
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), []
    
    row = dfa.iloc[0]
    gdc.set_primary_site(psi_id = row.pid)

    lista = [x for x in os.listdir(gdc.root_data) if f'_{gdc.psi_id}' in x]

    print(gdc.psi_id, len(lista))

    for fname in lista:
        filename_ori = gdc.root_data / fname
        filename_dest = gdc.root_psi / fname

        shutil.move(filename_ori, filename_dest)


for i, row in df_psi.iterrows():
    primary_site = row.primary_site
    print(">>>", primary_site)

    move_tables(primary_site)
